# Multi-Chain Gibbs Sampler

This notebook is the interactive companion to `run_multichains.py`. It uses the same public API rather than duplicating sampler setup logic in notebook cells.

Current scope from `run_multichains.py`:
- Full BDR models for D = 1, 2, 3, and 5 with layers 1, 2, and 3.
- W variants for layers 1, 2, and 3: `W_Known`, `No_W`, and `No_W_Selective`.
- SVD-based initialization helpers for `M`, `Lambda`, `V`, `W`, and matrix-Langevin priors.
- Optional R/coda parameter diagnostics through `compute_parameter_diagnostics=True`.
- Optional M/V backend selection with `mv_sampler='python'` or `mv_sampler='rstiefel'`.
- Sample saving and diagnostic plotting controlled by `save_samples`, `save_plots`, and `output_dir`.

Use a Python 3.9+ kernel. Some project modules use built-in generic annotations such as `list[str]`, which fail under Python 3.7.


**Setup**

Run this cell first. It adds the repository module folders to `sys.path`, sets a writable matplotlib cache inside the workspace, and imports the current API from `run_multichains.py`.


In [ ]:
# Keep annotation behavior consistent across Python versions used by notebook kernels.
from __future__ import annotations

# Standard-library tools used for path setup and API introspection.
import inspect
import os
import sys
from pathlib import Path

# Numpy is used for array shape checks and fixed column selections below.
import numpy as np

# Resolve paths relative to the notebook working directory.
BASE_DIR = Path.cwd()

# Matplotlib may need a writable cache when the repo is opened from a restricted environment.
CACHE_DIR = BASE_DIR / "test_outputs" / ".cache"
MPL_CACHE = CACHE_DIR / "matplotlib"
MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE))

# The project is organized as plain module folders, not an installed package.
# Add each folder that contains importable modules used by run_multichains.py.
for folder in [
    "Multichain",
    "Gibbs Sampling",
    "Parameter Sampler",
    "BDR Metrics and Plot",
    "Data Generation",
    "Covariance Functions",
]:
    folder_path = str(BASE_DIR / folder)
    if folder_path not in sys.path:
        sys.path.insert(0, folder_path)

# Data generators used in the small examples below.
from Data_generation import generate_case1_1d, generate_case1_2d

# Import the public functions and constants from run_multichains.py.
from run_multichains import (
    CONFIG_FUNCTIONS,
    LAMBDA_GAMMA_RATE,
    LAMBDA_GAMMA_SHAPE,
    compute_multichain_parameter_diagnostics,
    create_config_D1_L1,
    create_config_D1_L2,
    create_config_D1_L3,
    create_config_D2_L1,
    create_config_D2_L2,
    create_config_D2_L3,
    create_config_D3_L1,
    create_config_D3_L2,
    create_config_D3_L3,
    create_config_D5_L1,
    create_config_D5_L2,
    create_config_D5_L3,
    create_config_L1_No_W,
    create_config_L1_No_W_Selective,
    create_config_L1_W_Known,
    create_config_L2_No_W,
    create_config_L2_No_W_Selective,
    create_config_L2_W_Known,
    create_config_L3_No_W,
    create_config_L3_No_W_Selective,
    create_config_L3_W_Known,
    get_config_for,
    get_default_config,
    initialize_M_Lambda_V_W_D1,
    initialize_M_Lambda_V_W_Dgeneral,
    parameter_diagnostics,
    run_multichain_analysis,
)

# Quick confirmation that the module imported and exposes the expected presets.
print("Imported run_multichains.py API")
print(f"Lambda prior: Gamma(shape={LAMBDA_GAMMA_SHAPE}, rate={LAMBDA_GAMMA_RATE})")
print(f"Preset full-model configurations: {sorted(CONFIG_FUNCTIONS)}")


## API Reference From The Module

The next cell introspects `run_multichains.py` directly. This keeps the notebook aligned with the module if function signatures change.


In [ ]:
# Keep one list of all config helpers so the notebook can print current signatures.
config_helpers = [
    create_config_D1_L1, create_config_D1_L2, create_config_D1_L3,
    create_config_D2_L1, create_config_D2_L2, create_config_D2_L3,
    create_config_D3_L1, create_config_D3_L2, create_config_D3_L3,
    create_config_D5_L1, create_config_D5_L2, create_config_D5_L3,
    create_config_L1_W_Known, create_config_L1_No_W, create_config_L1_No_W_Selective,
    create_config_L2_W_Known, create_config_L2_No_W, create_config_L2_No_W_Selective,
    create_config_L3_W_Known, create_config_L3_No_W, create_config_L3_No_W_Selective,
]

# Print signatures directly from Python rather than duplicating them in markdown.
for fn in config_helpers:
    print(f"{fn.__name__}{inspect.signature(fn)}")

# Show the main runner signature because this is the function most examples call.
print("\nrun_multichain_analysis signature:")
print(inspect.signature(run_multichain_analysis))


## SVD Initialization

`run_multichains.py` initializes `M`, `Lambda`, `V`, `W`, `prior_M`, `prior_V`, and `prior_Lambda`. D=1 uses `initialize_M_Lambda_V_W_D1`; D>1 uses `initialize_M_Lambda_V_W_Dgeneral`.


In [ ]:
# Use p=10 because the included Case 1 synthetic generators have 10 input columns.
p = 10

# D=1 and D>1 use separate initialization helpers in run_multichains.py.
init_d1 = initialize_M_Lambda_V_W_D1(p=p, D=1, seed=42)
init_d2 = initialize_M_Lambda_V_W_Dgeneral(p=p, D=2, seed=42)

# Print shapes for each initialized matrix/prior so dimensional expectations are visible.
for label, init in [("D=1", init_d1), ("D=2", init_d2)]:
    print(f"\n{label}")
    for key in ["M_init", "Lambda_init", "V_init", "W_init", "prior_M", "prior_V", "prior_Lambda"]:
        value = np.asarray(init[key])
        print(f"  {key:<12} shape={value.shape}")


## Full-Model Configurations

Preset helpers are available through explicit functions and through `get_config_for(D, layer, **overrides)`.

Notes from the current module:
- D1, D2, and D3 preset helpers accept `p` and `seed` for automatic SVD initialization.
- D5 helpers accept `**kwargs`; pass explicit initial values if you need SVD initialization there.
- Some D>1 layer presets rely on `run_multichain_analysis()` defaults unless `kernel_type` is supplied, so the examples pass `kernel_type='separable_squared_exponential'` explicitly.


In [ ]:
# Generate a tiny Case 1a dataset for config and smoke-run examples.
data_d1 = generate_case1_1d(n=24, seed=42)
p_d1 = data_d1["X_train"].shape[1]

# Build a D=1, layer-1 full-model config with automatic SVD initialization.
config_d1_l1 = create_config_D1_L1(
    p=p_d1,
    seed=42,
    n_chains=2,
    n_iterations=2,
    burn_in=0,
    thin=1,
    save_samples=False,
    save_plots=False,
    compute_parameter_diagnostics=False,
    output_dir="test_outputs/notebook_D1_L1",
)

# Inspect the most important run-control and model-selection fields.
print("D=1, layer=1 config")
for key in ["D", "layer", "n_chains", "n_iterations", "burn_in", "thin", "kernel_type", "use_tf_gradients"]:
    print(f"  {key}: {config_d1_l1.get(key)}")
print(f"  W_init shape: {np.asarray(config_d1_l1['W_init']).shape}")

# Build a D=2, layer-2 config. The separable kernel is explicit for D>1.
config_d2_l2 = create_config_D2_L2(
    p=p_d1,
    seed=42,
    n_chains=2,
    n_iterations=2,
    burn_in=0,
    thin=1,
    kernel_type="separable_squared_exponential",
    save_samples=False,
    save_plots=False,
    compute_parameter_diagnostics=False,
    output_dir="test_outputs/notebook_D2_L2",
)

# D>1 theta initials should be vectors with length D.
print("\nD=2, layer=2 config")
for key in ["D", "layer", "kernel_type", "use_tf_gradients", "theta_y_init", "theta_q_init"]:
    print(f"  {key}: {config_d2_l2.get(key)}")


## Full-Model Test

This cell runs one very small D=1, layer-1 full model and spells out the main `run_multichain_analysis()` inputs.

Why this choice:
- `D=1` and `layer=1` are the simplest full BDR model, so this is the fastest sanity check.
- `n_chains=2`, `n_iterations=2`, `burn_in=0`, and `thin=1` produce exactly two saved samples per chain for a smoke run.
- `save_samples=False`, `save_plots=False`, and `compute_parameter_diagnostics=False` avoid extra files and optional R/coda requirements.
- The SVD-initialized `W_init`, `M_init`, `V_init`, `Lambda_init`, `prior_M`, and `prior_V` are reused from `config_d1_l1`.

To choose other options, change the explicit arguments below or build another config with `create_config_D*_L*()` / `get_config_for()`. Common changes are `D`, `layer`, `kernel_type`, `n_iterations`, `burn_in`, `thin`, `use_mle_all`, `save_samples`, `save_plots`, and `output_dir`.


In [ ]:
# Choose the response and predictor arrays. These must use matching train/test splits.
Y_train = data_d1["y_train"]
X_train = data_d1["X_train"]
Y_test = data_d1["y_test"]
X_test = data_d1["X_test"]

# Choose model structure and MCMC controls for this smoke run.
# For a real run, increase n_iterations, set burn_in > 0, and usually use 3+ chains.
D = config_d1_l1["D"]
layer = config_d1_l1["layer"]
n_chains = config_d1_l1["n_chains"]
n_iterations = config_d1_l1["n_iterations"]
burn_in = config_d1_l1["burn_in"]
thin = config_d1_l1["thin"]

# Choose estimation and kernel options. D=1 uses the isotropic kernel by default.
kernel_type = config_d1_l1["kernel_type"]
use_mle_all = config_d1_l1["use_mle_all"]
use_tf_gradients = config_d1_l1["use_tf_gradients"]

# Choose output behavior. These are off for a notebook smoke run.
save_samples = config_d1_l1["save_samples"]
save_plots = config_d1_l1["save_plots"]
compute_parameter_diagnostics = config_d1_l1["compute_parameter_diagnostics"]
output_dir = config_d1_l1["output_dir"]

# Run the tiny full model with explicit inputs so each choice is visible.
results_d1_l1 = run_multichain_analysis(
    # Data inputs.
    Y_train=Y_train,
    X_train=X_train,
    Y_test=Y_test,
    X_test=X_test,

    # Model selection.
    D=D,
    layer=layer,
    kernel_type=kernel_type,

    # MCMC controls.
    n_chains=n_chains,
    n_iterations=n_iterations,
    burn_in=burn_in,
    thin=thin,

    # Estimation controls.
    use_mle_all=use_mle_all,
    use_tf_gradients=use_tf_gradients,

    # SVD/matrix-Langevin initialization from create_config_D1_L1(..., p=p_d1).
    W_init=config_d1_l1["W_init"],
    M_init=config_d1_l1["M_init"],
    V_init=config_d1_l1["V_init"],
    Lambda_init=config_d1_l1["Lambda_init"],
    prior_M=config_d1_l1["prior_M"],
    prior_V=config_d1_l1["prior_V"],

    # Output and optional diagnostics.
    output_dir=output_dir,
    save_samples=save_samples,
    save_plots=save_plots,
    compute_parameter_diagnostics=compute_parameter_diagnostics,
    verbose=False,
)

# Confirm that the run produced two chains and two saved samples per chain.
print(f"chains: {len(results_d1_l1['chains_samples'])}")
print(f"saved samples in chain 1: {len(results_d1_l1['chains_samples'][0]['tau2_y'])}")
print(f"metric keys: {sorted(results_d1_l1['metrics_summary'])}")


## Reading Results

Full-model samplers generally use lowercase metric keys; variant samplers generally use uppercase metric keys. The helper below reads either form.


In [ ]:
# Full-model metrics use lowercase names, while variant metrics may use uppercase names.
# This helper lets later cells read either convention.
def metric_mean(results: dict, name: str):
    metrics = results.get("metrics_summary", {})
    metric = metrics.get(name, metrics.get(name.lower(), metrics.get(name.upper())))
    if isinstance(metric, dict):
        return metric.get("mean")
    return None

# Print the standard scalar metrics returned by the full-model smoke run.
for metric in ["rmspe", "nsme", "crps", "bic", "mlppd", "cp", "alci"]:
    print(f"{metric.upper():<6} {metric_mean(results_d1_l1, metric)}")

# Show sampled parameter names for the first chain.
print("\nSample keys in chain 1:")
print(sorted(results_d1_l1["chains_samples"][0]))

# Show metrics that are stored per saved posterior sample.
print("\nPer-sample metric keys in chain 1:")
print(sorted(k for k in results_d1_l1["chains_metrics"][0] if k.lower().endswith("_samples")))


## W-Variant Configurations

Variant helpers are available for all three layers:
- `W_Known`: pass a fixed projection matrix with shape `(p, D)`.
- `No_W`: use `X` directly and do not sample W/M/Lambda/V.
- `No_W_Selective`: use selected columns of `X`; pass `D` and optional `column_indices`.

The `create_config_L*_...` helpers return sampler-level dictionaries. They are useful references for defaults, but `run_multichain_analysis()` expects runner-level argument names and maps those into the variant samplers internally. The smoke run below therefore passes runner-level arguments directly.


In [ ]:
# The synthetic generator returns the true projection matrix, useful for W_Known examples.
true_w = data_d1["W"]

# These helper configs document variant defaults. They are sampler-level dictionaries.
variant_helper_configs = [
    create_config_L1_W_Known(W_fixed=true_w),
    create_config_L2_No_W(),
    create_config_L3_No_W_Selective(D=1, column_indices=np.array([0])),
]

# Print a compact summary of each variant helper config.
for config in variant_helper_configs:
    print(
        f"layer={config['layer']} variant={config['variant']} "
        f"D={config.get('D')} kernel={config.get('kernel_type')} "
        f"keys={sorted(config)[:6]}..."
    )


## Layer 1 W_Known Variant Test

This cell runs the layer-1 `W_Known` variant using the true synthetic projection matrix.


In [ ]:
# Run a tiny layer-1 W_Known variant through run_multichain_analysis.
# Unlike the helper configs above, these are runner-level keyword arguments.
results_l1_w_known = run_multichain_analysis(
    Y_train=data_d1["y_train"],
    X_train=data_d1["X_train"],
    Y_test=data_d1["y_test"],
    X_test=data_d1["X_test"],
    D=true_w.shape[1],
    layer=1,
    variant="W_Known",
    W_fixed=true_w,
    n_chains=2,
    n_iterations=2,
    burn_in=0,
    thin=1,
    kernel_type="isotropic_squared_exponential",
    save_samples=False,
    save_plots=False,
    compute_parameter_diagnostics=False,
    verbose=False,
)

# Check basic output shape and metric access for the variant result format.
print(f"chains: {len(results_l1_w_known['chains_samples'])}")
print(f"saved samples in chain 1: {len(results_l1_w_known['chains_samples'][0]['tau2_y'])}")
print(f"RMSPE mean: {metric_mean(results_l1_w_known, 'rmspe')}")
print(f"sample keys: {sorted(results_l1_w_known['chains_samples'][0])}")


## D>1 Configuration Example

For D>1 configurations, use vector theta initials and a separable kernel. TensorFlow gradients are enabled by default in many D>1 presets, but you can override `use_tf_gradients`. The next cell builds and inspects a D=2, layer-1 config; it does not run sampling.


In [ ]:
# Generate a tiny Case 1b dataset with a two-dimensional true subspace.
data_d2 = generate_case1_2d(n=24, seed=42)
p_d2 = data_d2["X_train"].shape[1]

# Build, but do not run, a D=2 full-model config using get_config_for.
config_d2_l1 = get_config_for(
    D=2,
    layer=1,
    p=p_d2,
    seed=42,
    n_chains=2,
    n_iterations=2,
    burn_in=0,
    thin=1,
    kernel_type="separable_squared_exponential",
    use_tf_gradients=True,
    save_samples=False,
    save_plots=False,
    compute_parameter_diagnostics=False,
    output_dir="test_outputs/notebook_D2_L1",
)

# Inspect D>1-specific fields: separable kernel, vector theta, and W_init shape.
print(f"D={config_d2_l1['D']}, layer={config_d2_l1['layer']}")
print(f"kernel_type={config_d2_l1.get('kernel_type')}")
print(f"theta_y_init={config_d2_l1.get('theta_y_init')}")
print(f"W_init shape={np.asarray(config_d2_l1['W_init']).shape}")


## Optional Parameter Diagnostics

`run_multichain_analysis()` can compute post-sampling parameter diagnostics with R/coda. This requires R, the R package `coda`, and `rpy2` in the Python environment. The next cell only builds an opt-in config; it does not run sampling.


In [ ]:
# This cell only builds a config. Run it later only if R, coda, and rpy2 are installed.
config_with_coda = create_config_D1_L1(
    p=p_d1,
    seed=42,
    n_chains=3,
    n_iterations=1000,
    burn_in=200,
    thin=2,
    compute_parameter_diagnostics=True,
    diagnostics_burn=200,
    diagnostics_ci=0.95,
    diagnostics_use_projection_for_W=False,
    diagnostics_parameters=None,
    save_samples=True,
    save_plots=True,
    output_dir="test_outputs/notebook_with_coda",
)

# Confirm the coda-related flags are present in the config.
print("Coda diagnostics enabled in config:", config_with_coda["compute_parameter_diagnostics"])
print("Diagnostics burn:", config_with_coda["diagnostics_burn"])


## Optional M/V Sampler Backend

The full model can update matrix-valued M and V using either the local Python implementation or R `rstiefel` through `rpy2`.


In [ ]:
# Build a config that uses the local Python matrix-von-Mises-Fisher updates for M/V.
config_python_mv = create_config_D2_L1(
    p=p_d2,
    seed=42,
    mv_sampler="python",
    kernel_type="separable_squared_exponential",
)

# Build a comparable config that would use R rstiefel through rpy2 if available.
config_rstiefel_mv = create_config_D2_L1(
    p=p_d2,
    seed=42,
    mv_sampler="rstiefel",
    rstiefel_rscol=1,
    kernel_type="separable_squared_exponential",
)

# Print backend selections without running either config.
print("Python M/V sampler:", config_python_mv["mv_sampler"])
print("rstiefel M/V sampler:", config_rstiefel_mv["mv_sampler"], "rscol=", config_rstiefel_mv["rstiefel_rscol"])


## Output Controls

Common output flags accepted by `run_multichain_analysis()`:
- `output_dir`: directory for diagnostics and saved samples.
- `save_samples`: writes `mcmc_samples.pkl` when true.
- `save_plots`: writes trace, density, autocorrelation, and prediction plots when true.
- `verbose`: prints sampler progress.

For quick notebook checks, use `save_samples=False`, `save_plots=False`, and `compute_parameter_diagnostics=False`. For real runs, increase MCMC settings and write outputs to a named folder under `simulation_outputs/` or `test_outputs/`.


In [ ]:
# Build a more realistic D=1, layer-2 config that writes samples and plots.
production_config = create_config_D1_L2(
    p=p_d1,
    seed=42,
    n_chains=3,
    n_iterations=2000,
    burn_in=500,
    thin=3,
    save_samples=True,
    save_plots=True,
    compute_parameter_diagnostics=False,
    output_dir="simulation_outputs/notebook_D1_L2",
)

# Print the output-related controls to verify where a real run would write files.
for key in ["n_chains", "n_iterations", "burn_in", "thin", "save_samples", "save_plots", "output_dir"]:
    print(f"{key}: {production_config[key]}")


## Summary

Use `run_multichains.py` as the source of truth for sampler setup. This notebook now imports and introspects that module directly, uses smoke-sized examples for interactive checks, and documents the current options for initialization, full models, W variants, optional R/coda diagnostics, M/V backends, and output controls.
